In [ ]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path

# Define paths
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data"
BREAD_PATH = DATA_PATH / "Bread"
PROCESSED_PATH = DATA_PATH / "processed"

PROCESSED_PATH.mkdir(exist_ok=True)

from src.data.preprocessing import prepare_sales_data_combined
from src.data.datasets import prepare_total_datasets, prepare_product_daily_dataset

In [ ]:
%run ./scripts/setup_core.sh
%run ./scripts/setup_autogluon.sh

In [ ]:
sales_daily, mismatch = prepare_sales_data_combined(BREAD_PATH)

print("Sales daily shape:", sales_daily.shape)
display(sales_daily.head())

sales_daily.to_parquet(PROCESSED_PATH / "sales_daily.parquet")
mismatch.to_parquet(PROCESSED_PATH / "promotion_mismatch.parquet")

In [ ]:
store_daily = prepare_total_datasets(sales_daily)

print("Store daily shape:", store_daily.shape)
display(store_daily.head())

store_daily.to_parquet(PROCESSED_PATH / "store_daily.parquet")

In [ ]:
share_daily = prepare_product_daily_dataset(sales_daily, target="share")
CATEGORICAL_COLS = ["day_of_week", "last_holiday_type", "next_holiday_type"]
for c in CATEGORICAL_COLS:
    if c in share_daily.columns:
        share_daily[c] = share_daily[c].astype("category")

print("Share daily shape:", share_daily.shape)
display(share_daily.head())
share_daily.to_parquet(PROCESSED_PATH / "share_daily.parquet")

In [ ]:
direct_daily = prepare_product_daily_dataset(sales_daily, target="quantity")
CATEGORICAL_COLS = ["day_of_week", "last_holiday_type", "next_holiday_type"]
for c in CATEGORICAL_COLS:
    if c in direct_daily.columns:
        direct_daily[c] = direct_daily[c].astype("category")

print("Direct daily shape:", direct_daily.shape)
display(direct_daily.head())
direct_daily.to_parquet(PROCESSED_PATH / "direct_daily.parquet")